This file contains a more automated approach to setting up or finding a wanted ec2 instance

In [ ]:
import boto3

ec2 = boto3.client('ec2')
iam = boto3.client('iam')

# Check if instance available

First, check if the instance with name *instance_name* is available

In [ ]:
response = ec2.describe_instances()

# instance_name = "mlops-mhofmann"
instance_name = "mlops-prod"
instance_id = None

for resp in response['Reservations']:
    resp = resp["Instances"][0]
    tags = resp.get("Tags", [])
    # print(tags)

    for tag in tags:
        if tag.get("Key") == "Name" and tag.get("Value") == instance_name:
            instance_id = resp["InstanceId"]
            print(f"Found instance {instance_name} with ID {instance_id}")

if not instance_id:
    print(f"Instance {instance_name} not found.")


It is possible to check the standard settings of AMIs:

In [ ]:
ec2.describe_images(ImageIds=["ami-0b293f83878b78ce7"])["Images"][0]["BlockDeviceMappings"]

If not available, create an instance with name *instance_name*

In [ ]:
if not instance_id:
    response = ec2.run_instances(
        ImageId="ami-0b293f83878b78ce7", #'ami-09f69fcf6c4c6d58b', # image optimized for deep learning
        MinCount=1,
        MaxCount=1,
        InstanceType='m7i-flex.large', 
        KeyName='mhofmann',
        BlockDeviceMappings=[
            {
                "DeviceName": "/dev/sda1",
                "Ebs": {
                    "DeleteOnTermination": True,
                    "VolumeSize": 120
                }
            }
        ]
    )
    instance_id = response['Instances'][0]['InstanceId']
    print(f"Created new instance {instance_name} with ID {instance_id}")
    ec2.create_tags(
        Resources=[instance_id],
        Tags=[{"Key": "Name", "Value": instance_name}]
    )
    print(f"Gave instance {instance_id} the name {instance_name}")
else:
    print(f"Instance {instance_name} already exists with ID {instance_id}")  


Every ec2 instance needs a security group for access management.\
The script down below gets the security group with name *security_group_name*.\
If the security group with according name is not available, create a new one with the according name.\

In [ ]:
security_group_name = "mhofmann"

response = ec2.describe_security_groups()

# get id of security group with name 'security_group_name'
security_group_id = [x["GroupId"] for x in response['SecurityGroups'] if x["GroupName"] == security_group_name]

if not security_group_name:
    response = ec2.create_security_group(
        GroupName=security_group_name,
        Description='New security group for MLOps course',
    )
    security_group_id = response['GroupId']
    print(f"Created security group {security_group_name} with ID {security_group_id}")
else:
    security_group_id = security_group_id[0]
    print(f"Found security group {security_group_name} with ID {security_group_id}")


Add the access permissions to the (empty) security group.

In [ ]:
from botocore.exceptions import ClientError

def update_security_group(group_id, protocol, port, cidr):
    try:
        response = ec2.authorize_security_group_ingress(
            GroupId=group_id,
            IpPermissions=[
                {
                    'IpProtocol': protocol,
                    'FromPort': port,
                    'ToPort': port,
                    'IpRanges': [{'CidrIp': cidr}]
                }
            ]
        )
        print(f"Updated security group {group_id} to allow {protocol} on port {port} from {cidr}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'InvalidPermission.Duplicate':
            print(f"Permission already exists for {protocol} on port {port} from {cidr}")
        else:
            print(f"Failed to update security group: {e}")

update_security_group(security_group_id, "tcp", 22, "0.0.0.0/0")  # SSH
update_security_group(security_group_id, "tcp", 80, "0.0.0.0/0")  # HTTP
update_security_group(security_group_id, "tcp", 8501, "0.0.0.0/0") 
update_security_group(security_group_id, "tcp", 8502, "0.0.0.0/0")

Apply the security group to the ec2 instance

In [ ]:
ec2.modify_instance_attribute(
    InstanceId=instance_id,
    Groups=[security_group_id])

To make the EC2 access the S3 buckets, use a new IAM role named *ec2-s3-full-access* and apply it to the instance.

In [ ]:
role_name = "ec2-s3-full-access"

response = iam.get_role(RoleName=role_name)

role_arn = response['Role']['Arn']

role_arn

# Ensure there is an instance profile with the same name as the role
instance_profile_name = role_name
try:
    iam.get_instance_profile(InstanceProfileName=instance_profile_name)
except iam.exceptions.NoSuchEntityException:
    # Create an instance profile if it doesn't exist
    iam.create_instance_profile(InstanceProfileName=instance_profile_name)
    # Add role to the instance profile
    iam.add_role_to_instance_profile(
        InstanceProfileName=instance_profile_name,
        RoleName=role_name
    )

# Attach the instance profile to the EC2 instance
ec2.associate_iam_instance_profile(
    IamInstanceProfile={
        'Name': instance_profile_name
    },
    InstanceId=instance_id
)

In [ ]:
role_name = "ec2-s3-full-access"

response = iam.get_role(RoleName=role_name)

role_arn = response['Role']['Arn']

instance_profile_name = role_name

try:
    response = iam.get_instance_profile(InstanceProfileName=instance_profile_name)
    print(f"Instance profile {instance_profile_name} already exists.")
except iam.exceptions.NoSuchEntityException:
    response = iam.create_instance_profile(InstanceProfileName=instance_profile_name)
    iam.add_role_to_instance_profile(InstanceProfileName=instance_profile_name, RoleName=role_name)
    print(f"Instance profile {instance_profile_name} created and role {role_name} added.")

ec2.associate_iam_instance_profile(
    IamInstanceProfile={
        'Name': instance_profile_name
    },
    InstanceId=instance_id
)
print(f"Instance profile {instance_profile_name} associated with instance {instance_id}.")


# Functions for checking status and terminate the instance

In [ ]:
import time

def wait_for_status(instance_id, target_status):
    while True:
        response = ec2.describe_instances(InstanceIds=[instance_id])
        status = response['Reservations'][0]['Instances'][0]['State']['Name']
        if status == target_status:
            print(f"Instance {instance_id} is now in '{target_status}' state.")
            break
        else:
            print(f"Instance {instance_id} is still in status '{status}'. Waiting for status '{target_status}'...")
        time.sleep(10)  # check every 10 seconds

def terminate_instance(instance_id):
    ec2.terminate_instances(InstanceIds=[instance_id])
    print(f"Terminating instance {instance_id}...")
    wait_for_status(instance_id, 'terminated')

In [ ]:
terminate_instance(instance_id)